# CS224N: Hugging Face Transformers 教程 (2026 冬季)
更新者：Minsik Oh

原作者：Ben Newman

感谢 Anna Goldie 的反馈！

本笔记本将介绍 Hugging Face Transformers Python 库以及一些可以用来发挥其优势的常见模式。它对于在您的项目中使用或微调预训练的 transformer 模型非常有用。


Hugging Face 提供了对模型（包括实现它们的代码及其预训练权重，包括最新的 LLM 如 Llama3、DBRX 等）、特定模型的 tokenizer、常见 NLP 任务的 pipeline，以及在独立的 `datasets` 包中的数据集和指标的访问。它在 PyTorch、Tensorflow 和 Flax 中都有实现（不过我们这里将使用 PyTorch 版本！）


我们将介绍几个使用案例：
* Tokenizers（分词器）和 Models（模型）概述
* 微调（Finetuning）- 针对您自己的任务。我们将使用一个情感分类的示例。


教授在上周四的讲座中谈到了几个主要的项目类型：
1. 将现有的预训练模型应用于新的应用程序或任务，并探索如何处理/解决它
2. 实现一个新的或复杂的神经网络架构，并展示其在某些数据上的性能
3. 分析模型的行为：它如何表示语言学知识，或者它可以处理哪些现象或它会犯哪些错误

在这些类型中，`transformers` 对 (1) 和 (3) 的帮助最大。(2) 涉及一定的学习曲线，但如果您掌握了它，您会发现基于 Huggingface 提供的现有模型来设计模型非常方便。我们在这里不作介绍，请参考 [这个示例](https://huggingface.co/docs/transformers/en/custom_models)。



以下是用于制作本教程的介绍该库的其他资源：

* [Hugging Face 文档](https://huggingface.co/docs/transformers/index)
  * 清晰的文档说明
  * 教程、逐步指南和示例笔记本
  * 可用模型列表
* [Hugging Face 课程](https://huggingface.co/course/)
* [Hugging Face 示例](https://github.com/huggingface/transformers/tree/main/examples/pytorch) 您可以在使用 Huggingface 的极其不同的下游任务/模型中找到非常相似的代码结构。
* [Hugging Face O'Reilly 书籍](https://www.oreilly.com/library/view/natural-language-processing/9781098136789/)



In [ ]:
!pip install transformers
!pip install datasets
!pip install accelerate

In [ ]:
from collections import defaultdict, Counter
import json

from matplotlib import pyplot as plt
import numpy as np
import torch

def print_encoding(model_inputs, indent=4):
    indent_str = " " * indent
    print("{")
    for k, v in model_inputs.items():
        print(indent_str + k + ":")
        print(indent_str + indent_str + str(v))
    print("}")

## Part 0: 使用 Hugging Face Transformers 的常见模式

我们将从 Hugging Face Transformers 的常见使用模式开始，以情感分析为例。

首先，在 [Hub](https://huggingface.co/models) 上寻找一个模型。任何人都可以上传自己的模型供其他人使用。（我正在使用的是来自 [这篇论文](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=3489963) 的情感分析模型）。

然后，需要初始化两个对象——一个 **tokenizer**（分词器）和一个 **model**（模型）

* Tokenizer 将字符串转换为模型所需的词表 ID 列表
* Model 接收这些词表 ID 并产生预测

![full_nlp_pipeline.png](https://huggingface.co/datasets/huggingface-course/documentation-images/resolve/main/en/chapter2/full_nlp_pipeline.svg)
来自 [https://huggingface.co/course/chapter2/2?fw=pt](https://huggingface.co/course/chapter2/2?fw=pt)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 初始化分词器 (tokenizer)
tokenizer = AutoTokenizer.from_pretrained("siebert/sentiment-roberta-large-english")
# 初始化模型 (model)
model = AutoModelForSequenceClassification.from_pretrained("siebert/sentiment-roberta-large-english")

In [ ]:
inputs = "I'm excited to learn about Hugging Face Transformers!"
tokenized_inputs = tokenizer(inputs, return_tensors="pt")
outputs = model(**tokenized_inputs)

labels = ['NEGATIVE', 'POSITIVE']
prediction = torch.argmax(outputs.logits)


print("Input:")
print(inputs)
print()
print("Tokenized Inputs:")
print_encoding(tokenized_inputs)
print()
print("Model Outputs:")
print(outputs)
print()
print(f"The prediction is {labels[prediction]}")

### 0.1 Tokenizers (分词器)

预训练模型会带有用于对其输入进行预处理的 **tokenizers**（分词器）。分词器接收原始字符串或字符串列表，并输出实际上包含模型输入的字典。


您可以使用特定于要使用的模型的分词器类（此处为 DistilBERT），或者使用 AutoTokenizer 类来访问分词器。
快速分词器（Fast Tokenizers）是用 Rust 编写的，而它们的慢速版本是用 Python 编写的。

In [ ]:
from transformers import DistilBertTokenizer, DistilBertTokenizerFast, AutoTokenizer
name = "distilbert/distilbert-base-cased"
# 从 Hub 加载时 name = "user/name"
# 使用 save_pretrained() 方法时 name = 本地路径

tokenizer = DistilBertTokenizer.from_pretrained(name)      # 用 Python 编写
print(tokenizer)
tokenizer = DistilBertTokenizerFast.from_pretrained(name)  # 用 Rust 编写
print(tokenizer)
tokenizer = AutoTokenizer.from_pretrained(name) # 很方便！默认为 Fast 版本
print(tokenizer)

In [ ]:
# 这是调用分词器的方法
input_str = "Hugging Face Transformers is great!"
tokenized_inputs = tokenizer(input_str) # https://huggingface.co/learn/nlp-course/en/chapter6/6


print("Vanilla Tokenization")
print_encoding(tokenized_inputs)
print()

# 两种访问方式：
print(tokenized_inputs.input_ids)
print(tokenized_inputs["input_ids"])

In [ ]:
cls = [tokenizer.cls_token_id]
sep = [tokenizer.sep_token_id]

# 分词过程分为几个步骤：
input_tokens = tokenizer.tokenize(input_str)
input_ids = tokenizer.convert_tokens_to_ids(input_tokens)
input_ids_special_tokens = cls + input_ids + sep

decoded_str = tokenizer.decode(input_ids_special_tokens)

print("start:                ", input_str)
print("tokenize:             ", input_tokens)
print("convert_tokens_to_ids:", input_ids)
print("add special tokens:   ", input_ids_special_tokens)
print("--------")
print("decode:               ", decoded_str)

# 注意：这些步骤不会创建 attention mask，也不会添加特殊字符

In [ ]:
# 对于快速分词器 (Fast Tokenizers)，还有另一个选项：
inputs = tokenizer._tokenizer.encode(input_str)

print(input_str)
print("-"*5)
print(f"Number of tokens: {len(inputs)}")
print(f"Ids: {inputs.ids}")
print(f"Tokens: {inputs.tokens}")
print(f"Special tokens mask: {inputs.special_tokens_mask}")
print()
print("char_to_word gives the wordpiece of a character in the input")
char_idx = 8
print(f"For example, the {char_idx + 1}th character of the string is '{input_str[char_idx]}',"+\
      f" and it's part of wordpiece {inputs.char_to_token(char_idx)}, '{inputs.tokens[inputs.char_to_token(char_idx)]}'")

In [ ]:
# 其他酷炫技巧：
# 分词器可以返回 pytorch 张量
model_inputs = tokenizer("Hugging Face Transformers is great!", return_tensors="pt")
print("PyTorch Tensors:")
print_encoding(model_inputs)

In [ ]:
# 您可以向分词器传入多个字符串，并根据需要对它们进行填充 (padding)
model_inputs = tokenizer(["Hugging Face Transformers is great!",
                         "The quick brown fox jumps over the lazy dog." +\
                         "Then the dog got up and ran away because she didn't like foxes.",
                         ],
                         return_tensors="pt",
                         padding=True,
                         truncation=True)
print(f"Pad token: {tokenizer.pad_token} | Pad token id: {tokenizer.pad_token_id}")
print("Padding:")
print_encoding(model_inputs)

In [ ]:
# 您还可以一次性解码整个批次：
print("Batch Decode:")
print(tokenizer.batch_decode(model_inputs.input_ids))
print()
print("Batch Decode: (no special characters)")
print(tokenizer.batch_decode(model_inputs.input_ids, skip_special_tokens=True))

欲了解有关分词器的更多信息，您可以查看：
[Hugging Face Transformers 文档](https://huggingface.co/docs/transformers/main_classes/tokenizer) 和 [Hugging Face Tokenizers 库](https://huggingface.co/docs/tokenizers/python/latest/quicktour.html)（针对快速分词器）。Tokenizers 库甚至允许您训练自己的分词器！

### 0.2 Models (模型)




初始化模型与初始化分词器非常相似。您既可以使用特定于您的模型的模型类，也可以使用 AutoModel 类。我更倾向于使用 AutoModel，尤其是当我想对比不同模型时，因为以字符串形式指定模型非常容易。

虽然大多数预训练的 transformer 具有相似的架构，但是如果您要进行序列分类、问答或其他任务，您就必须训练一些额外的权重，称为“头”（heads）。当您指定模型类时，Hugging Face 会自动设置您需要的架构。例如，我们正在做情感分析，因此我们将使用 `DistilBertForSequenceClassification`。如果我们打算在 DistilBERT 的掩码语言建模（masked-language modeling）训练目标上继续训练它，我们将使用 `DistilBertForMaskedLM`；如果我们只是想要模型的表征（可能用于我们自己的下游任务），我们可以只使用 `DistilBertModel`。


这里是一张精美的模型结构插图，重新制作自此处的插图：[https://huggingface.co/course/chapter2/2?fw=pt](https://huggingface.co/course/chapter2/2?fw=pt)。
![model_illustration.png](https://huggingface.co/datasets/huggingface-course/documentation-images/resolve/main/en/chapter2/transformer_and_head.svg)


以下是一些示例：
```
*Model
*ForMaskedLM
*ForSequenceClassification
*ForTokenClassification
*ForQuestionAnswering
*ForMultipleChoice
...
```
其中 `*` 可以是 `AutoModel` 或特定的预训练模型（例如 `DistilBert`）


模型有三种类型：
* 编码器（Encoders，例如 BERT）
* 解码器（Decoders，例如 GPT2）
* 编码器-解码器模型（Encoder-Decoder 模型，例如 BART 或 T5）

您可用的特定于任务的类取决于您所处理的模型类型。


完整的选项列表可以在 [文档](https://huggingface.co/docs/transformers/model_doc/auto) 中找到。请注意，并非所有模型都与所有模型架构兼容，例如 DistilBERT 与 Seq2Seq 模型不兼容，因为它只包含一个编码器。


In [ ]:
from transformers import AutoModelForSequenceClassification, DistilBertForSequenceClassification, DistilBertModel
print('Loading base model')
base_model = DistilBertModel.from_pretrained('distilbert-base-cased')
print("Loading classification model from base model's checkpoint")
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-cased', num_labels=2)
model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-cased', num_labels=2)


您还可以使用随机权重进行初始化

In [ ]:
from transformers import DistilBertConfig, DistilBertModel

# 初始化一个 DistilBERT 配置
configuration = DistilBertConfig()
configuration.num_labels=2
# 根据该配置初始化一个模型（带有随机权重）
model = DistilBertForSequenceClassification(configuration)

# 访问模型配置
configuration = model.config

我们在此处收到了一个警告，因为序列分类参数尚未训练。

向模型传递输入非常简单。它们接受关键字参数作为输入

In [ ]:
model_inputs = tokenizer(input_str, return_tensors="pt")

# 选项 1
model_outputs = model(input_ids=model_inputs.input_ids, attention_mask=model_inputs.attention_mask)

# 选项 2 - 分词器返回的字典键与模型期望的关键字参数相同
#            

# f({k1: v1, k2: v2}) = f(k1=v1, k2=v2)

model_outputs = model(**model_inputs)

print(model_inputs)
print()
print(model_outputs)
print()
print(f"Distribution over labels: {torch.softmax(model_outputs.logits, dim=1)}")

如果您注意到了的话，对于二分类任务，我们有两个类别这看起来有点奇怪——您完全可以只用一个类别然后选择一个阈值。这是因为 Hugging Face 模型计算损失的方式。这会增加我们拥有的参数数量，但在其他方面应该不会影响性能。

这些模型只是 PyTorch 的 Module！您可以使用您的 `loss_func` 来计算损失并调用 `loss.backward`。您可以使用您之前使用过的任何优化器或学习率调度器

In [ ]:
# 您可以像往常一样计算损失
label = torch.tensor([1])
loss = torch.nn.functional.cross_entropy(model_outputs.logits, label)
print(loss)
loss.backward()

# 您可以获取参数
list(model.named_parameters())[0]

Hugging Face 也提供了一种额外的计算损失的简便方法：

In [ ]:
# 为了计算损失，我们需要传入 label：
model_inputs = tokenizer(input_str, return_tensors="pt")

labels = ['NEGATIVE', 'POSITIVE']
model_inputs['labels'] = torch.tensor([1])

model_outputs = model(**model_inputs)


print(model_outputs)
print()
print(f"Model predictions: {labels[model_outputs.logits.argmax()]}")

最后需要注意的一点是——您可以非常轻松地从模型中获取隐藏状态（hidden states）和注意力权重（attention weights）。如果您正在进行分析性项目，这会特别有帮助。（例如，参见 [BERT 都在看什么？](https://arxiv.org/abs/1906.04341））。

In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained("distilbert-base-cased", output_attentions=True, output_hidden_states=True)
model.eval()

model_inputs = tokenizer(input_str, return_tensors="pt")
with torch.no_grad():
    model_output = model(**model_inputs)


print("Hidden state size (per layer):  ", model_output.hidden_states[0].shape)
print("Attention head size (per layer):", model_output.attentions[0].shape)     # (layer, batch, query_word_idx, key_word_idxs)
                                                                               # y轴是 query，x轴是 key
# print(model_output)

In [ ]:
tokens = tokenizer.convert_ids_to_tokens(model_inputs.input_ids[0])
print(tokens)


n_layers = len(model_output.attentions)
n_heads = len(model_output.attentions[0][0])
fig, axes = plt.subplots(6, 12)
fig.set_size_inches(18.5*2, 10.5*2)
for layer in range(n_layers):
    for i in range(n_heads):
        axes[layer, i].imshow(model_output.attentions[layer][0, i])
        axes[layer][i].set_xticks(list(range(9)))
        axes[layer][i].set_xticklabels(labels=tokens, rotation="vertical")
        axes[layer][i].set_yticks(list(range(9)))
        axes[layer][i].set_yticklabels(labels=tokens)

        if layer == 5:
            axes[layer, i].set(xlabel=f"head={i}")
        if i == 0:
            axes[layer, i].set(ylabel=f"layer={layer}")

plt.subplots_adjust(wspace=0.3)
plt.show()

## Part 1: 微调 (Finetuning)

对于您的项目，您很可能希望微调一个预训练模型。这稍微复杂一些，但仍然非常简单。

### 2.1 加载数据集

除了模型之外，[Hub](https://huggingface.co/datasets) 上也提供了许多数据集。

In [ ]:
from datasets import load_dataset, DatasetDict



# DataLoader(zip(list1, list2))
dataset_name = "stanfordnlp/imdb"

imdb_dataset = load_dataset(dataset_name)


# 为了速度以及能够在 CPU 上运行，仅保留前 50 个 token
def truncate(example):
    return {
        'text': " ".join(example['text'].split()[:50]),
        'label': example['label']
    }

imdb_dataset

In [ ]:

# 随机选取 128 个样本用于训练，32 个样本用于验证
small_imdb_dataset = DatasetDict(
    train=imdb_dataset['train'].shuffle(seed=1111).select(range(128)).map(truncate),
    val=imdb_dataset['train'].shuffle(seed=1111).select(range(128, 160)).map(truncate),
)
small_imdb_dataset

In [ ]:
small_imdb_dataset['train'][:10]

In [ ]:
# 准备数据集 - 这里以 16 个样本为一批次进行分词处理。
small_tokenized_dataset = small_imdb_dataset.map(
    lambda example: tokenizer(example['text'], padding=True, truncation=True), # https://huggingface.co/docs/transformers/pad_truncation
    batched=True,
    batch_size=16
)

small_tokenized_dataset = small_tokenized_dataset.remove_columns(["text"])
small_tokenized_dataset = small_tokenized_dataset.rename_column("label", "labels")
small_tokenized_dataset.set_format("torch")

In [ ]:
small_tokenized_dataset['train'][0:2]

In [ ]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(small_tokenized_dataset['train'], batch_size=16)
eval_dataloader = DataLoader(small_tokenized_dataset['val'], batch_size=16)

### 2.2 训练

为了训练您的模型，您可以直接使用与 PyTorch 中相同的训练循环。Hugging Face 模型也是 `torch.nn.Module`，因此反向传播以相同的方式进行，您甚至可以使用相同的优化器。Hugging Face 还包含了用于训练 Transformer 模型的优化器和学习率调度器，所以您也可以使用它们。

对于优化，我们使用的是 AdamW 优化器，它与 Adam 几乎完全相同，只是它还包含了权重衰减 (weight decay)。
我们还使用了一个线性学习率调度器，它在训练过程中每个训练步骤后都会稍微降低学习率。

还有其他可以使用的优化器和学习率调度器，但这些是默认的。如果您想探索，可以查看 [Hugging Face 提供的](https://huggingface.co/docs/transformers/main_classes/optimizer_schedules#schedules) 那些，或者通过 [PyTorch](https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate) 提供的那些（例如 [ReduceLROnPlateau](https://pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.ReduceLROnPlateau.html)，它仅在验证损失停止下降时降低学习率），或者编写您自己的（就像作业 4 中的那样）。

In [ ]:
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
from tqdm.notebook import tqdm


model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-cased', num_labels=2)

num_epochs = 1
num_training_steps = len(train_dataloader)
optimizer = AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
lr_scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

best_val_loss = float("inf")
progress_bar = tqdm(range(num_training_steps))
for epoch in range(num_epochs):
    # 训练阶段
    model.train()
    for batch_i, batch in enumerate(train_dataloader):

        # batch = ([text1, text2], [0, 1])

        output = model(**batch)

        optimizer.zero_grad()
        output.loss.backward()
        optimizer.step()
        lr_scheduler.step()
        progress_bar.update(1)

    # 验证阶段
    model.eval()
    for batch_i, batch in enumerate(eval_dataloader):
        with torch.no_grad():
            output = model(**batch)
        loss += output.loss

    avg_val_loss = loss / len(eval_dataloader)
    print(f"Validation loss: {avg_val_loss}")
    if avg_val_loss < best_val_loss:
        print("Saving checkpoint!")
        best_val_loss = avg_val_loss
        # torch.save({
        #     'epoch': epoch,
        #     'model_state_dict': model.state_dict(),
        #     'optimizer_state_dict': optimizer.state_dict(),
        #     'val_loss': best_val_loss,
        #     },
        #     f"checkpoints/epoch_{epoch}.pt"
        # )

In [ ]:
batch['input_ids'].max()

虽然您可以像我们在作业 4 中所做的那样使用 PyTorch 来训练您的模型，但 Hugging Face 提供了一个功能强大的 `Trainer` 类来处理大多数需求。我认为它非常好用，不过我还是推荐进行一些自定义设置。

In [ ]:
imdb_dataset = load_dataset("stanfordnlp/imdb")

small_imdb_dataset = DatasetDict(
    train=imdb_dataset['train'].shuffle(seed=1111).select(range(128)).map(truncate),
    val=imdb_dataset['train'].shuffle(seed=1111).select(range(128, 160)).map(truncate),
)

small_tokenized_dataset = small_imdb_dataset.map(
    lambda example: tokenizer(example['text'], padding='max_length', truncation=True, max_length=512),
    batched=True,
    batch_size=16
)

`TrainingArguments` 指定了不同的训练参数，例如评估和保存模型检查点的频率、保存它们的位置等。您可以自定义的方面**非常多**，非常值得去[这里](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.TrainingArguments)查看。您可以控制的一些参数包括：
* 学习率 (learning rate)、权重衰减 (weight decay)、梯度裁剪 (gradient clipping)，
* 检查点保存、日志记录和评估频率
* 记录日志的目标位置（默认为 TensorBoard，但如果您使用 WandB 或 MLFlow，它们也提供了集成方式）

`Trainer` 实际执行训练。您可以将 `TrainingArguments`、模型、数据集、分词器、优化器，甚至用于恢复训练的模型检查点传递给它。`compute_metrics` 函数在评估/验证结束时调用，以计算评估指标。

In [ ]:
from transformers import TrainingArguments, Trainer

model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-cased', num_labels=2)

arguments = TrainingArguments(
    output_dir="sample_hf_trainer",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    eval_strategy="epoch", # 在每个 epoch 结束时进行验证
    save_strategy="epoch",
    learning_rate=2e-5,
    load_best_model_at_end=True,
    seed=224
)


def compute_metrics(eval_pred):
    """验证结束时调用。给出准确率"""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    # 计算准确率
    return {"accuracy": np.mean(predictions == labels)}


trainer = Trainer(
    model=model,
    args=arguments,
    train_dataset=small_tokenized_dataset['train'],
    eval_dataset=small_tokenized_dataset['val'], # 在进行最终评估时请更改为 test！
    compute_metrics=compute_metrics
)

#### Callbacks（回调）：日志记录与早停（Early Stopping）


如果您希望在训练期间的不同时间点（例如在评估之后或在一个 epoch 结束之后）触发某些操作，Hugging Face Transformers 还允许您编写 `Callbacks`。例如，有一个专门用于早停的回调，并且我通常还会编写一个用于日志记录的回调。

有关回调的更多信息，请参见[这里](https://huggingface.co/docs/transformers/main_classes/callback#transformers.TrainerCallback)。

In [ ]:
from transformers import TrainerCallback, EarlyStoppingCallback

class LoggingCallback(TrainerCallback):
    def __init__(self, log_path):
        self.log_path = log_path
    # 会在每个日志记录步骤调用 on_log，该步骤由 TrainerArguments 指定（即 TrainerArguments.logging_steps）
    def on_log(self, args, state, control, logs=None, **kwargs):
        _ = logs.pop("total_flos", None)
        if state.is_local_process_zero:
            with open(self.log_path, "a") as f:
                f.write(json.dumps(logs) + "\n")
    # def on_epoch(...)


trainer.add_callback(EarlyStoppingCallback(early_stopping_praise=1, early_stopping_threshold=0.0))
trainer.add_callback(LoggingCallback("sample_hf_trainer/log.jsonl"))

In [ ]:
# 训练模型
trainer.train()

In [ ]:
# 评估模型非常简单

# results = trainer.evaluate()                           # 仅获取评估指标
results = trainer.predict(small_tokenized_dataset['val']) # 同时也返回预测结果

In [ ]:
results

In [ ]:
# 要加载我们保存的模型，我们可以将检查点路径传递给 `from_pretrained` 方法：
test_str = "I enjoyed the movie!"

finetuned_model = AutoModelForSequenceClassification.from_pretrained("sample_hf_trainer/checkpoint-8")
model_inputs = tokenizer(test_str, return_tensors="pt")
prediction = torch.argmax(finetuned_model(**model_inputs).logits)
print(["NEGATIVE", "POSITIVE"][prediction])

此处还包括一些微调的实用建议：

**优秀的默认超参数。** 您要使用的超参数取决于您的任务和数据集。您应该进行超参数搜索以找到最佳超参数。尽管如此，这里提供了一些微调时不错的初始值：
* Epochs（迭代轮数）：{2, 3, 4}（数据量越大，所需的 epoch 越少）
* Batch size（批次大小）：（越大越好：在显存允许的情况下尽可能大）
* Optimizer（优化器）：AdamW
* AdamW 学习率：{2e-5, 5e-5}
* Learning rate scheduler（学习率调度器）：在训练的前 {0, 100, 500} 步进行线性预热 (linear warm up)
* weight_decay（权重衰减，即 L2 正则化）：{0, 0.01, 0.1}

您应该监控验证损失来判定何时找到了理想的超参数。

我们可以集成到 Trainer 中的内容还有很多，可以让它更加实用，包括日志记录、保存模型检查点等！您甚至可以编写它的子类来添加您自己的个性化组件。您可以查看 [此链接](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.Trainer) 了解更多有关 Trainer 的信息。

## Appendix 0: 文本生成 (Generation)

在上面的例子中，我们在分类任务上微调了模型，但您也可以在生成任务上微调模型。`generate` 函数可以轻松地让这些模型生成内容。例如：

In [ ]:
from transformers import AutoModelForCausalLM

gpt2_tokenizer = AutoTokenizer.from_pretrained('gpt2')

gpt2 = AutoModelForCausalLM.from_pretrained('distilgpt2')
gpt2.config.pad_token_id = gpt2.config.eos_token_id  # 防止解码时出现警告

In [ ]:
prompt = "Once upon a time"

tokenized_prompt = gpt2_tokenizer(prompt, return_tensors="pt")

for i in range(10):
    output = gpt2.generate(**tokenized_prompt,
                  max_length=50,
                  do_sample=True,
                  top_p=0.9)

    print(f"{i + 1}) {gpt2_tokenizer.batch_decode(output)[0]}")

## Appendix 1: 定义自定义数据集

定义数据集有几种方法，但我将展示一个使用 PyTorch Dataloaders 的示例。这个示例使用了一个编码器-解码器数据集，即 [E2E 数据集](https://arxiv.org/abs/1706.09254)，它将关于餐馆的结构化信息映射为自然语言描述。

我们使用在 [tuetschek/e2e_nlg](https://huggingface.co/datasets/tuetschek/e2e_nlg/tree/refs%2Fconvert%2Fparquet) HuggingFace 数据集中找到的自动转换的 Parquet 文件。您可以在这篇 [HuggingFace 文档](https://huggingface.co/docs/dataset-viewer/en/parquet) 中了解该过程。

In [ ]:
# 选项 1：加载到 Hugging Face Datasets

import pandas as pd
from datasets import Dataset

df = pd.read_parquet("hf://datasets/tuetschek/e2e_nlg@refs%2Fconvert%2Fparquet/default/train/0000.parquet")
custom_dataset = Dataset.from_pandas(df)

In [ ]:
import pandas as pd
from torch.utils.data import Dataset

class E2EDataset(Dataset):
    """当我们调用 __getitem__ 时对数据进行分词"""
    def __init__(self, path, tokenizer):
        df = pd.read_parquet(path)
        self.data = df[["human_reference", "meaning_representation"]].to_dict(orient="records")
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        inputs = self.tokenizer(self.data[i]["human_reference"])
        labels = self.tokenizer(self.data[i]["meaning_representation"])
        inputs["labels"] = labels.input_ids
        return inputs

In [ ]:
bart_tokenizer = AutoTokenizer.from_pretrained('facebook/bart-base')

In [ ]:
dataset = E2EDataset("hf://datasets/tuetschek/e2e_nlg@refs%2Fconvert%2Fparquet/default/train/0000.parquet", bart_tokenizer)

In [ ]:
bart_tokenizer(text=["This is the first test.", "This is the second test."], text_target=["Target 1", "Target 2"], return_tensors="pt", padding=True, truncation=True)

In [ ]:
dataset[0]

## Appendix 2: Pipelines (流水线)

对于一些标准的 NLP 任务（如情感分类或问答），在 Hugging Face Transformer 的 [_Pipeline_](https://huggingface.co/docs/transformers/v4.16.2/en/main_classes/pipelines#transformers.pipeline) 接口中已经有预训练（且微调过！）的模型可用。

对于您的项目，您可能不会频繁使用它，但了解一下仍然很有必要！

这里有一个情感分析的例子：

In [ ]:
from transformers import pipeline

sentiment_analysis = pipeline("sentiment-analysis", model="siebert/sentiment-roberta-large-english")

您只需在字符串上调用 pipeline 即可运行它

In [ ]:
sentiment_analysis("Hugging Face Transformers is really cool!")

或者运行在字符串列表上：

In [ ]:
sentiment_analysis(["I didn't know if I would like Hákarl, but it turned out pretty good.",
                    "I didn't know if I would like Hákarl, and it was just as bad as I'd heard."])

您可以在[这里](https://huggingface.co/docs/transformers/main_classes/pipelines)找到关于 pipeline 的更多信息（包括哪些 pipeline 是可用的）

## Appendix 4: 掩码语言建模 (Masked Language Modeling)

In [ ]:
from transformers import AutoModelForMaskedLM

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased", fast=True)
bert = AutoModelForMaskedLM.from_pretrained("bert-base-cased")

In [ ]:
prompt = "I am [MASK] to learn about HuggingFace!"
model = pipeline("fill-mask", "bert-base-cased")
model(prompt)

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt")
mask_index = np.where(inputs['input_ids'] == tokenizer.mask_token_id)
outputs = bert(**inputs)
top_5_predictions = torch.softmax(outputs.logits[mask_index], dim=1).topk(5)

print(prompt)
for i in range(5):
    prediction = tokenizer.decode(top_5_predictions.indices[0, i])
    prob = top_5_predictions.values[0, i]
    print(f"  {i+1}) {prediction}\t{prob:.3f}")